In [18]:
import pandas as pd
import numpy as np

from utils import Display

In [19]:
housing = pd.read_csv('processed_data/01_housing.csv')
park = pd.read_csv('processed_data/01_park.csv')

In [20]:
Display(housing)

Shape: (261159, 35)
Index(['城市', '区域', '板块', '小区名称', 'price', 'unit_price', '链家编号', '看房时间', '房屋户型',
       '所在楼层', '建筑面积', '房屋朝向', '建筑结构', '装修情况', '梯户比例', '配备电梯', '别墅类型', '挂牌时间',
       '交易权属', '上次交易', '房屋用途', '房屋年限', '产权所属', '房本备件', '房源标签', '核心卖点', '户型介绍',
       '周边配套', '交通出行', 'x', 'y', '年份', 'nearest_park_dist_km',
       'nearest_park_index', 'nearest_park_industry'],
      dtype='object')


,Missing,Unique,Dtype
城市,0,1,object
区域,0,17,object
板块,0,255,object
小区名称,0,8024,object
price,0,2818,float64
unit_price,0,99366,object
链家编号,0,261159,float64
看房时间,0,4,object
房屋户型,18573,549,object
所在楼层,16788,234,object


【col: 城市】
  - 北京: 261159 次
----------------------------------------
【col: 区域】
  - 朝阳: 58859 次
  - 丰台: 29637 次
  - 海淀: 24887 次
  - 通州: 21044 次
  - 大兴: 20854 次
----------------------------------------
【col: 板块】
  - 长阳: 6974 次
  - 回龙观: 5837 次
  - 顺义城: 5390 次
  - 望京: 5380 次
  - 良乡: 4173 次
----------------------------------------
【col: 小区名称】
  - 远洋山水: 531 次
  - 荣丰2008: 527 次
  - 天通苑东一区: 523 次
  - 龙湖长城源著2号院: 517 次
  - 芍药居北里: 494 次
----------------------------------------
【col: price】
  - 450.0: 1974 次
  - 550.0: 1686 次
  - 420.0: 1667 次
  - 320.0: 1663 次
  - 330.0: 1647 次
----------------------------------------
【col: unit_price】
  - 100000元/平米: 112 次
  - 50000元/平米: 43 次
  - 125000元/平米: 41 次
  - 45455元/平米: 39 次
  - 62500元/平米: 37 次
----------------------------------------
【col: 链家编号】
  - 101091978877.0: 1 次
  - 101118266406.0: 1 次
  - 101118266246.0: 1 次
  - 101118266261.0: 1 次
  - 101118266294.0: 1 次
----------------------------------------
【col: 看房时间】
  - 提前预约随时可看: 182031 次
  - 有租户需预约: 5677

In [21]:
housing.columns

Index(['城市', '区域', '板块', '小区名称', 'price', 'unit_price', '链家编号', '看房时间', '房屋户型',
       '所在楼层', '建筑面积', '房屋朝向', '建筑结构', '装修情况', '梯户比例', '配备电梯', '别墅类型', '挂牌时间',
       '交易权属', '上次交易', '房屋用途', '房屋年限', '产权所属', '房本备件', '房源标签', '核心卖点', '户型介绍',
       '周边配套', '交通出行', 'x', 'y', '年份', 'nearest_park_dist_km',
       'nearest_park_index', 'nearest_park_industry'],
      dtype='object')

In [ ]:
# plot the distribution of price & unit price
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.hist(housing['unit_price'], bins=50, alpha=0.7)
plt.xlabel('Log Price')
plt.ylabel('Frequency')
plt.title('Distribution of Price')
plt.show()

/home/yubo/.conda/envs/Housing/lib/python3.10/site-packages/IPython/core/pylabtools.py:170: UserWarning: Glyph 20803 (\N{CJK UNIFIED IDEOGRAPH-5143}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/home/yubo/.conda/envs/Housing/lib/python3.10/site-packages/IPython/core/pylabtools.py:170: UserWarning: Glyph 24179 (\N{CJK UNIFIED IDEOGRAPH-5E73}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)
/home/yubo/.conda/envs/Housing/lib/python3.10/site-packages/IPython/core/pylabtools.py:170: UserWarning: Glyph 31859 (\N{CJK UNIFIED IDEOGRAPH-7C73}) missing from font(s) DejaVu Sans.
  fig.canvas.print_figure(bytes_io, **kw)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder,StandardScaler

city_le = LabelEncoder()
dist_le = LabelEncoder()
neigh_le = LabelEncoder()

housing['city_idx'] = city_le.fit_transform(housing['城市'])
housing['dist_idx'] = dist_le.fit_transform(housing['区域'])
housing['neigh_idx'] = neigh_le.fit_transform(housing['板块'])

n_cities = len(city_le.classes_)   
n_dists = len(dist_le.classes_)
n_neighs = len(neigh_le.classes_)

In [ ]:
city_tensor = torch.tensor(housing['city_idx'].values, dtype=torch.long)
dist_tensor = torch.tensor(housing['dist_idx'].values, dtype=torch.long)
neigh_tensor = torch.tensor(housing['neigh_idx'].values, dtype=torch.long)
target_tensor = torch.tensor(housing['log_price'].values, dtype=torch.float32)

class EmbeddingModel(nn.Module):
    def __init__(self, n_cities, n_dists, n_neighs, city_dim=8, dist_dim=24, neigh_dim=32):
        super().__init__()
        self.city_emb = nn.Embedding(n_cities, city_dim)
        self.dist_emb = nn.Embedding(n_dists, dist_dim)
        self.neigh_emb = nn.Embedding(n_neighs, neigh_dim)
        self.net = nn.Sequential(
            nn.Linear(city_dim + dist_dim + neigh_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.1),
            
            nn.Linear(64, 1)
        )
        
    def forward(self, c_idx, d_idx, n_idx):
        c = self.city_emb(c_idx)
        d = self.dist_emb(d_idx)
        n = self.neigh_emb(n_idx)
        out = torch.cat([c, d, n], dim=1)
        return self.net(out)
    


KeyError: 'log_price'